Creating white keys through a loop that creates 14 rectangles with a certain starting point (based on the white key width) and the height of the full piano. In order to label each key accordingly, creating a list with each white key note and also cycling through these labels in the loop.

In [22]:
import tkinter as tk
root = tk.Tk()

label = tk.Label(root, text="Piano GUI")
label.pack()

canvas = tk.Canvas(root, width = 1120, height = 250, bg = "gray")
canvas.pack()

white_notes = ["C", "D", "E", "F", "G", "A", "B", "C", "D", "E", "F", "G", "A", "B"]

white_key_width = 80
white_key_height = 250

for i in range(14):
    x1 = i * white_key_width
    x2 = x1 + white_key_width
    y1 = 0
    y2 = white_key_height
    canvas.create_rectangle(x1, y1, x2, y2, fill="white", outline="black", tags=(f"white{i}", "white_key"))
    #the "tags=" assign each rectangle to a specific name that is "white1", "white2"

    text_x = x1 + (white_key_width / 2)
    text_y = white_key_height - 25
    canvas.create_text(text_x, text_y, text=white_notes[i], fill="black")


Using the seam of each white key as a midpoint for the black key dimensions. Couldn't do a simple loop like the white keys because there is a 2-3 pattern. 

In [23]:
def blk_key_arr(midp_num, note_name):
    mid_x = midp_num * white_key_width
    
    x1 = mid_x - 25
    x2 = mid_x + 25
    y1 = 0
    y2 = 150
    canvas.create_rectangle(x1, y1, x2, y2, fill="black", outline="black", tags=(f"black{midp_num}", "black_key")) 
    #the "tags=" assign each rectangle to a specific name that is "black1", "black2" (number depending on midp_num param in the class)

    canvas.create_text(mid_x, 100, text=note_name, fill="white")

blk_key_arr(1, "C#/Db")
blk_key_arr(2, "D#/Eb")

blk_key_arr(4, "F#/Gb")
blk_key_arr(5, "G#/Ab")
blk_key_arr(6, "A#/Bb")

blk_key_arr(8, "C#/Db")
blk_key_arr(9, "D#/Eb")

blk_key_arr(11, "F#/Gb")
blk_key_arr(12, "G#/Ab")
blk_key_arr(13, "A#/Bb")

Creating the legend for what color corresponds to which chord member.

In [24]:
leg_frame = tk.Frame(root, bg = 'white', borderwidth=1, relief = 'solid')
leg_frame.pack(pady = 8)

red_leg = tk.Label(leg_frame, text = "red: root", fg = 'red', bg = 'white')
red_leg.pack()

orange_leg = tk.Label(leg_frame, text = "orange: 3rd", fg = 'orange', bg = 'white')
orange_leg.pack()

purp_leg = tk.Label(leg_frame, text = "purple: 5th", fg = 'purple', bg = 'white')
purp_leg.pack()

pink_leg = tk.Label(leg_frame, text = "pink: 7th", fg = 'magenta', bg = 'white')
pink_leg.pack()

In [25]:
note_to_tags = {
    'C': ['white0', 'white7'],
    'C#/Db': ['black1', 'black8'],
    'D': ['white1', 'white8'],
    'D#/Eb': ['black2', 'black9'],
    'E': ['white2', 'white9'],
    'F': ['white3', 'white10'],
    'F#/Gb': ['black4', 'black11'],
    'G': ['white4', 'white11'],
    'G#/Ab': ['black5', 'black12'],
    'A': ['white5', 'white12'],
    'A#/Bb': ['black6', 'black13'],
    'B': ['white6', 'white13'],
}

Making chord formulas using number of semitones between the notes in the chromatic scale. 

In [26]:
chrom_scale = ["C", "C#/Db", "D", "D#/Eb", "E", "F", "F#/Gb", "G", "G#/Ab", "A", "A#/Bb", "B"]

def chord_formulas(root, quality, inversion):
    formulas = {
        'Major triad':[0, 4, 7], 
        'Minor triad':[0, 3, 7], 
        'Diminished triad':[0, 3, 6],
        'Major 7th': [0, 4, 7, 11],
        'Minor 7th': [0, 3, 7, 10],
        'Dominant 7th/Major Minor 7th': [0, 4, 7, 10],
        'Half Diminished 7th': [0, 3, 6, 10]
    }

    root_ind = chrom_scale.index(root)
    chord_spell = [chrom_scale[(root_ind + num) % 12] for num in formulas[quality]]

    inversion_map = {
        'root position': chord_spell,
        '1st inversion': chord_spell[1:] + chord_spell[:1],
        '2nd inversion': chord_spell[2:] + chord_spell[:2],
        '3rd inversion': chord_spell[3:] + chord_spell[:3]
    }

    return inversion_map[inversion]
    

Each of these functions is part of the way the keys get colored, taking input from the later coded dropdown menus.

In [27]:
def reset_colors():
    canvas.itemconfig("white_key", fill="white")
    canvas.itemconfig("black_key", fill="black")
    #resets key colors to default before highlighting next input

def highlight_note():
    reset_colors()
    input_note = note_opt.get().strip()
    
    if input_note in note_to_tags:
        for tag in note_to_tags[input_note]:
            canvas.itemconfig(tag, fill="orange")

def plot_chord():
    reset_colors()
    user_root = root_opt.get()
    user_quality = qual_opt.get()
    user_inversion = inv_opt.get()

    notes_in_chord = chord_formulas(user_root, user_quality, user_inversion)
    root_ind = chrom_scale.index(user_root)

    oct_ind = 0
    last_sc_ind = -1

    for note in notes_in_chord:
        current_ind = chrom_scale.index(note)
        if current_ind < last_sc_ind:
            oct_ind = 1
        last_sc_ind = current_ind

        semitones = (current_ind - root_ind) % 12

        tag = note_to_tags[note][oct_ind]
        
        if semitones == 0:
            color = "red"
            canvas.itemconfig(tag, fill=color)
        elif semitones == 3 or semitones == 4:
            color = "orange"
            canvas.itemconfig(tag, fill=color)
        elif semitones == 6 or semitones == 7:
            color = "purple"
            canvas.itemconfig(tag, fill=color)
        else:
            color = "magenta"
            canvas.itemconfig(tag, fill=color)

This block creates the dropdown menus to make the use experience less buggy. The change_inversion function is there so that when the user picks a triad, the 3rd inversion option is automatically disabled.

In [28]:
root_opts = chrom_scale
quality_opts = ['Major triad','Minor triad','Diminished triad','Major 7th','Minor 7th','Dominant 7th/Major Minor 7th','Half Diminished 7th']
inversion_opts = ['root position', '1st inversion', '2nd inversion', '3rd inversion']

root_opt = tk.StringVar(value = 'C')
tk.OptionMenu(root, root_opt, *root_opts).pack(side= tk.LEFT, padx=5)

qual_opt = tk.StringVar(value = 'Major triad')
tk.OptionMenu(root, qual_opt, *quality_opts).pack(side= tk.LEFT, padx=5) 

inv_opt = tk.StringVar(value = 'root position')
inv_menu = tk.OptionMenu(root, inv_opt, *inversion_opts)
inv_menu.pack(side=tk.LEFT, padx=5)

def change_inversion(*args):
    if 'triad' in qual_opt.get():
        inv_menu["menu"].entryconfig(3, state="disabled")
        if inv_opt.get() == '3rd inversion':
            inv_opt.set('root position')
    else:
        inv_menu["menu"].entryconfig(3, state="normal")
        
qual_opt.trace_add("write", change_inversion)

change_inversion()

tk.Button(root, text='Plot Chord', command = plot_chord).pack(side= tk.LEFT, padx=5)

note_opts = chrom_scale

note_opt = tk.StringVar(value = 'C')
tk.OptionMenu(root, note_opt, *note_opts).pack(side= tk.RIGHT, padx=5)

submit_btn = tk.Button(root, text="Highlight note", command=highlight_note)
submit_btn.pack(side = tk.RIGHT, padx = 5)

root.mainloop()